# Bundle Anatomy

Builds a two-member bundle and verifies member naming, baseline metadata, and lazy comparison surfaces.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.ModelLog.is_recurrent",
    "tl.ModelLog.keep_unsaved_layers",
    "tl.ModelLog.last_run_ctx",
    "tl.ModelLog.last_run_records",
    "tl.ModelLog.layer_dict_all_keys",
    "tl.ModelLog.layer_dict_main_keys",
    "tl.ModelLog.layer_labels",
    "tl.ModelLog.layer_labels_no_pass",
    "tl.ModelLog.layer_labels_w_pass",
    "tl.ModelLog.layer_list",
    "tl.ModelLog.layer_logs",
    "tl.ModelLog.layer_num_passes",
    "tl.ModelLog.layers",
    "tl.ModelLog.layers_with_params",
    "tl.ModelLog.layers_with_saved_activations",
    "tl.ModelLog.layers_with_saved_gradients",
    "tl.ModelLog.load",
    "tl.ModelLog.log_backward",
    "tl.ModelLog.logging_mode",
    "tl.ModelLog.macs_by_type",
    "tl.ModelLog.manual_tensor_connections",
    "tl.ModelLog.mark_input_output_distances",
    "tl.ModelLog.max_recurrent_loops",
    "tl.ModelLog.model_name",
    "tl.ModelLog.module_filter_fn",
    "tl.ModelLog.modules",
    "tl.ModelLog.name",
    "tl.ModelLog.num_context_lines",
    "tl.ModelLog.num_grad_fns",
    "tl.ModelLog.num_intervening_grad_fns",
    "tl.ModelLog.num_operations",
    "tl.ModelLog.num_streamed_passes",
    "tl.ModelLog.num_tensors_saved",
    "tl.ModelLog.num_tensors_total",
    "tl.ModelLog.observer_spans",
    "tl.ModelLog.operation_history",
    "tl.ModelLog.orphan_layers",
    "tl.ModelLog.output_device",
    "tl.ModelLog.output_layers",
    "tl.ModelLog.param_logs",
    "tl.ModelLog.params",
    "tl.ModelLog.parent_run",
    "tl.ModelLog.pass_end_time",
    "tl.ModelLog.pass_start_time",
    "tl.ModelLog.preview_fastlog",
    "tl.ModelLog.raise_on_nan",
    "tl.ModelLog.random_seed_used",
    "tl.ModelLog.recording_backward",
    "tl.ModelLog.recording_kept",
    "tl.ModelLog.relationship_evidence",
    "tl.ModelLog.release_param_refs",
    "tl.ModelLog.remove",
    "tl.ModelLog.render_dagua_graph",
    "tl.ModelLog.render_graph",
    "tl.ModelLog.replace_run_state_from",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")